# 21 · FEC in optical standards

Optical standards fix their FEC, and with it the **pre-FEC BER threshold**:
the raw error rate the link must reach for the decoder to deliver a
practically error-free output (1e-12 to 1e-15).  This notebook measures
the thresholds of the codes in use:

1. IM/DD Ethernet: KP4 RS(544, 514) and the KP4 + Hamming(128, 120)
   concatenation of IEEE 802.3dj;
2. coherent hard-decision FEC: the G.709 RS(255, 239) and the G.709.2
   staircase code;
3. 400ZR: the concatenated C-FEC (staircase + soft Hamming(128, 119));
4. PON: the RS(248, 216) of XGS-PON and the LDPC(17280, 14592) of 50G-PON.

Hard-decision codes see only bit errors, so a binary symmetric channel
(random flips) measures them; soft-decision ones need LLRs from a
modulated signal.  It builds on [07 · Channel coding](07-channel-coding.ipynb).

In [ ]:
from math import comb

import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device
from commkit.coding import (
    Chase,
    Concatenated,
    LDPCCode,
    ReedSolomonCode,
    Stages,
    StaircaseCode,
    Syndrome,
    decode,
    encode,
)

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)
RNG = np.random.default_rng(0)


def host(a):
    return to_device(a, "cpu")


def hard_post_fec(code, p, num_codewords):
    """Post-FEC BER of a hard-decision code at pre-FEC BER p (random flips)."""
    u = RNG.integers(0, 2, (num_codewords, code.k), dtype=np.int8)
    words = host(encode(ck.backend.to_device(u, DEVICE), code=code))
    flipped = words ^ (RNG.random(words.shape) < p).astype(np.int8)
    res = decode(ck.backend.to_device(flipped, DEVICE), code=code)
    return float(np.mean(host(res.bits) != u))


def soft_link(code, constellation, esn0_db, num_codewords, method=None, seed=1):
    """Pre-FEC BER and post-FEC BER of a soft-decoded code over AWGN."""
    m = constellation.bits_per_symbol
    tx = ck.generate(constellation, num_codewords * code.n // m, symbol_rate=1e9,
                     code=code, rng=seed, device=DEVICE)  # fmt: skip
    rx = ck.impairments.apply_awgn(tx, esn0_db=esn0_db, rng=seed + 1)
    llrs = ck.mapping.compute_llr(rx, noise_var=10 ** (-esn0_db / 10))
    res = decode(llrs, code=code, method=method)
    return ck.metrics.ber(rx), ck.metrics.ber(res.bits, tx.reference.info_bits)

## 1. IM/DD Ethernet: KP4 and its concatenation

KP4 corrects 15 of the 544 10-bit symbols of a codeword.  Its frames fail
only with more than 15 symbol errors, so the output error rate at the
threshold follows from the binomial tail, far below what can be simulated:

In [ ]:
def kp4_frame_error(p_bit):
    p = 1 - (1 - p_bit) ** 10
    return sum(comb(544, i) * p**i * (1 - p) ** (544 - i) for i in range(16, 545))


kp4 = ReedSolomonCode.kp4()
for p in (1e-4, 2.4e-4, 5e-4, 1e-3):
    print(f"pre-FEC BER {p:.1e}: frame error rate {kp4_frame_error(p):.1e}")
print(f"simulated at 3e-3: post-FEC BER {hard_post_fec(kp4, 3e-3, 200):.1e}")

The 200G/lane PMDs of IEEE 802.3dj add an inner extended Hamming(128, 120)
code, decoded softly, under KP4.  With PAM4 over AWGN, the concatenation
decodes a raw BER of 3e-3, where KP4 alone fails, and breaks down near
6.5e-3:

In [ ]:
dj = Concatenated.ieee_802_3dj()
pam4 = Constellation.pam(4)
print(dj, f"rate {dj.rate:.3f}")
method = Stages(inner=Chase(3, hard=Syndrome(max_weight=1)))
for esn0 in (13.5, 14.5, 15.5):
    pre, post = soft_link(dj, pam4, esn0, num_codewords=6, method=method)
    print(f"Es/N0 {esn0} dB: pre-FEC BER {pre:.1e}, post-FEC BER {post:.1e}")

## 2. Coherent hard-decision FEC: RS and staircase

The G.709 RS(255, 239) of the first 10G and 100G systems corrects 8 bytes
per codeword.  The staircase code of G.709.2, at the same 6.7% overhead,
couples BCH codewords across blocks and decodes iteratively.  Its
threshold is an order of magnitude higher: G.709 specifies RS(255, 239)
down to 1.8e-4 for an output BER of 1e-15, G.709.2 the staircase code
down to 4.5e-3.  Simulated, RS(255, 239) still fails at 3e-3 where the
staircase code corrects everything at 4e-3:

In [ ]:
rs = ReedSolomonCode.g709()
for p in (1e-3, 3e-3):
    print(
        f"RS(255, 239) at pre-FEC BER {p:.0e}: post-FEC BER {hard_post_fec(rs, p, 2000):.1e}"
    )
sc = StaircaseCode.g709_2(2)
for p in (4.0e-3, 7.0e-3):
    print(
        f"staircase at pre-FEC BER {p:.1e}: post-FEC BER {hard_post_fec(sc, p, 1):.1e}"
    )

## 3. 400ZR: the concatenated C-FEC

400ZR puts the staircase code around a soft-decoded double-extended
Hamming(128, 119), exactly as OIF-400ZR-01.0 defines it, for 14.8%
overhead (a little more here, from the frame's terminating block).  OIF
gives a threshold of 1.25e-2.  With 16-QAM:

In [ ]:
cfec = Concatenated.cfec(2)
method = Stages(inner=Chase(4, hard=Syndrome(max_weight=1)))
for esn0 in (13.2, 13.7):
    pre, post = soft_link(
        cfec, Constellation.qam(16), esn0, num_codewords=1, method=method
    )
    print(f"Es/N0 {esn0} dB: pre-FEC BER {pre:.2e}, post-FEC BER {post:.1e}")

## 4. PON: RS and LDPC

XGS-PON uses RS(248, 216), shortened from RS(255, 223).  The 25G/50G PON
standards moved to the LDPC(17280, 14592) of IEEE 802.3ca and ITU-T
G.9804.2, decoded softly:

In [ ]:
xgs = ReedSolomonCode.xgspon()
for p in (1e-3, 3e-3):
    print(
        f"RS(248, 216) at pre-FEC BER {p:.0e}: post-FEC BER {hard_post_fec(xgs, p, 2000):.1e}"
    )
pon = LDPCCode.pon()
for esn0 in (5.5, 6.0):
    pre, post = soft_link(pon, Constellation.qam(4), esn0, num_codewords=8)
    print(
        f"LDPC(17280, 14592) at Es/N0 {esn0} dB: pre-FEC BER {pre:.2e}, post-FEC BER {post:.1e}"
    )

**Summary**

| standard | code | overhead | threshold (pre-FEC BER) |
| --- | --- | --- | --- |
| IEEE 802.3 KP4 | RS(544, 514) | 5.8% | 2.4e-4 |
| IEEE 802.3dj | KP4 + Hamming(128, 120), soft inner | 12.4% | about 5e-3 |
| ITU-T G.709 | RS(255, 239) | 6.7% | 1.8e-4 |
| ITU-T G.709.2 | staircase, BCH(1022, 990) | 6.7% | 4.5e-3 |
| OIF 400ZR | staircase + Hamming(128, 119), soft inner | 14.8% | 1.25e-2 |
| ITU-T G.9807.1 | RS(248, 216) | 14.8% | about 1e-3 |
| ITU-T G.9804.2 | LDPC(17280, 14592), soft | 18.4% | about 2e-2 |

The thresholds of the hard-decision codes hold for output error rates of
1e-12 to 1e-15, far below what a notebook can simulate, so their simulated
waterfalls sit above them; the soft-decoded and iteratively decoded codes
fall within a few frames, and the measured points bracket them.  Soft
decoding and iterative decoding of coupled codes buy the higher thresholds
of the recent standards.

**Next:** [22 · End-to-end coherent transceiver](22-coherent-transceiver-end-to-end.ipynb).